In [ ]:
import pandas as pd
import numpy as np
import xarray as xr

In [ ]:
def extract_station_coords(dat_file):
    """Read station elevations and coordinates from text file."""
    with open(dat_file, 'r') as f:
        f.readline() # skip title
        header = [next(f).split()[4:] for _ in range(4)]
    stations = pd.DataFrame(np.array(header[:-1]).T, index=header[-1], columns=['z','x','y'])
    return stations

# example of reading station coordinates of radiation input data
dat_file = './Rhone/Input/timeseries/radiation.dat'
stations = extract_station_coords(dat_file)

In [ ]:
def read_meteo_station_inputs(dat_file):
    "read meteo input data from a .dat file"
    df0 = pd.read_csv(dat_file, sep="\t", header=4, na_values=-9999.)
    data = df0.iloc[:, 4:].apply(pd.to_numeric, errors='coerce')
    time = df0.iloc[:, :4].astype(int)
    dt = pd.DataFrame({
        'year': time['YY'],
        'month': time['MM'],
        'day': time['DD']
    })
    data.loc[:, 'datetime'] = pd.to_datetime(dt)
    data['datetime'] = data['datetime'] + pd.to_timedelta(time['HH'].astype(int), unit='h')
    data.set_index('datetime', inplace=True)
    data.index.name = 'time'

    with open(dat_file, 'r') as f:
        header = [next(f) for _ in range(4)]
    return data, header

# example of reading radiation input data
dat_file = './Rhone/Input/timeseries/radiation.dat'
rad, _ = read_meteo_station_inputs(dat_file)

In [ ]:
def read_fluxes(dat_file):
    "read WASIM fluxes output statistics: discharge, precipitation, etc."
    df0 = pd.read_csv(dat_file, sep="\t", header=0, low_memory=False)
    df0_data = df0.iloc[2:, 4:].apply(pd.to_numeric, errors='coerce')
    df0_time = df0.iloc[2:, :4].astype(int)
    dt_input = pd.DataFrame({
        'year': df0_time['YY'],
        'month': df0_time['MM'],
        'day': df0_time['DD']
    })
    df0_data.loc[:, 'datetime'] = pd.to_datetime(dt_input) 
    df0_data['datetime'] = df0_data['datetime'] + pd.to_timedelta(df0_time['HH'].astype(int), unit='h')
    df0_data.set_index('datetime', inplace=True)
    df0_data.index.name = 'time'
    df_stacked = df0_data.stack()
    df_stacked.index.names = ['time', 'region']
    return df_stacked

# example of reading discharge outputs
dat_file = './Rhone/Output/qgko.dat'
discharge, _ = read_fluxes(dat_file)

In [ ]:
def read_esri_ascii(file_path):
    with open(file_path, 'r') as f:
            header = {}
            # Read the first 6 lines for the header
            for _ in range(6):
                line = f.readline().strip().split()
                header[line[0].lower()] = float(line[1])

            # Extract header information
            ncols = int(header['ncols'])
            nrows = int(header['nrows'])
            xllcorner = header['xllcorner']
            yllcorner = header['yllcorner']
            cellsize = header['cellsize']
            nodata_value = header['nodata_value']

            # Read the remaining lines as data
            data = np.loadtxt(f)
    return data, ncols, nrows, xllcorner, yllcorner, cellsize, nodata_value

def create_xy_coordinates_from_ascii(xllcorner, yllcorner, ncols, nrows, cellsize, cell_center=True):
    if cell_center:
        center_adjust = cellsize/2
        xll = xllcorner + center_adjust; yll = yllcorner + center_adjust
        x = np.arange(xll, xll + ncols * cellsize, cellsize)
        y = np.arange(yll, yll + nrows * cellsize, cellsize)
    else:
        x = np.arange(xllcorner, xllcorner + ncols * cellsize, cellsize)
        y = np.arange(yllcorner, yllcorner + nrows * cellsize, cellsize)
    return x, y

# Example of reading a grid ascii file and into a DataArray
grid_file = './Rhone/Input/grids/r500.ezg'
data, ncols, nrows, xllcorner, yllcorner, cellsize, nodata_value = read_esri_ascii(grid_file)
x_center, y_center = create_xy_coordinates_from_ascii(xllcorner, yllcorner, ncols, nrows, cellsize, cell_center=True)
x_corner, y_corner = create_xy_coordinates_from_ascii(xllcorner, yllcorner, ncols, nrows, cellsize, cell_center=False)
data = np.flipud(data)
da_grid = xr.DataArray(data, dims=('y','x'), coords={'x':x_center, 'y':y_center})
da_grid = da_grid.where(da_grid!=-9999.)